In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("../data/processed/churn_clean_raw.csv")
df.shape

(1500, 21)

In [3]:
binary_cols = [
    "Gender", "Senior Citizen", "Partner", "Dependents", "Phone Service",
    "Multiple Lines", "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies", "Paperless Billing",
]
for col in binary_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

df = pd.get_dummies(df, columns=["Internet Service", "Contract", "Payment Method"], drop_first=True)
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)
df.columns = [c.replace(" ", "_") for c in df.columns]
df.shape

(1500, 25)

##### Business features
- `Customer_Lifetime_Months` - tenure, renamed for clarity
- `Average_Monthly_Spend` - total charges / tenure
- `Service_Count` - how many of the 6 add-ons they use
- `High_Monthly_Charge` - flag for top 25% spenders
- `Tenure_Group` - New / Established / Loyal buckets

In [4]:
df["Customer_Lifetime_Months"] = df["Tenure_Months"]
df["Average_Monthly_Spend"] = df["Total_Charges"] / df["Tenure_Months"].replace(0, 1)

service_cols = ["Online_Security", "Online_Backup", "Device_Protection",
                "Tech_Support", "Streaming_TV", "Streaming_Movies"]
df["Service_Count"] = df[service_cols].sum(axis=1)

df["High_Monthly_Charge"] = (df["Monthly_Charges"] > df["Monthly_Charges"].quantile(0.75)).astype(int)

df["Tenure_Group"] = pd.cut(df["Tenure_Months"], bins=[-1, 12, 48, 1000], labels=["New", "Established", "Loyal"])
df = pd.get_dummies(df, columns=["Tenure_Group"], drop_first=True)
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

df.shape

(1500, 31)

In [5]:
df.head()

,Gender,Senior_Citizen,Partner,Dependents,Tenure_Months,Phone_Service,Multiple_Lines,Online_Security,Online_Backup,Device_Protection,...,Contract_Two_Year,Payment_Method_Credit_Card_(automatic),Payment_Method_Electronic_Check,Payment_Method_Mailed_Check,Customer_Lifetime_Months,Average_Monthly_Spend,Service_Count,High_Monthly_Charge,Tenure_Group_Established,Tenure_Group_Loyal
0,1,0,0,0,32,1,2,0,2,2,...,1,0,0,1,32,104.043750,6,1,1,0
1,0,0,0,0,1,1,2,0,0,2,...,0,0,1,0,1,75.510000,4,0,0,0
2,0,0,0,1,49,1,2,2,0,0,...,0,0,1,0,49,32.358367,6,0,0,1
3,1,0,0,0,33,1,2,2,0,2,...,0,1,0,0,33,92.926061,8,0,1,0
4,1,0,1,1,56,1,2,2,2,0,...,0,0,1,0,56,64.252679,6,0,0,1


In [6]:
df.to_csv("../data/processed/churn_features.csv", index=False)
print("Saved.")

Saved.
